GitHub Vulnerability Scraper

This Jupyter notebook is designed to scrape and process vulnerability data from the NIST National Vulnerability Database (NVD) and GitHub. It fetches CVE details and commit information to analyze vulnerabilities and their impact.

## Prerequisites
- Ensure you have a GitHub token saved in a `.env` file as `GITHUB_TOKEN`.
- Install the necessary libraries using the provided `pip` install command.

## Key Variables
- `GITHUB_TOKEN`: Token for authenticating with GitHub API.
- `BASE_URL`: Base URL for the NIST API.

## Functions
- `is_github`: Checks if a URL is a GitHub URL.
- `url_type`: Determines the type of GitHub URL (commit, pull, etc.).
- `call_api`: Calls the NIST API to fetch vulnerability data.
- `get_commit_info`: Gets commit info from GitHub API.
- `get_commit_info_from_url`: Extracts commit info from a GitHub URL.
- `parse_github_commit_url`: Parses a GitHub commit URL.
- `parse_vulnerability`: Parses vulnerability data to extract relevant information.
- `scrape_between`: Scrapes data for a specific CWE ID between given dates.
- `generate_monthly_ranges_pd`: Generates monthly date ranges for scraping.
- `load_existing_data`: Loads existing data from a CSV file.

## Steps
1. Define the start and end dates for scraping.
2. Define the CWE IDs to scrape.
3. Load existing data to avoid duplicate entries.
4. Scrape data for each CWE ID and save the results to a CSV file.

In [ ]:
%pip install requests pandas python-dotenv --quiet

In [9]:
import requests
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv
import json
import time
from urllib.parse import urlparse
from pathlib import Path

# Load environment variables
env_path = Path('..') / '.env'
load_dotenv()

# GitHub token
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

# Define base URL for the NIST API
BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

## Functions to execute call to NIST API

In [ ]:
def is_github(url):
    return "github.com" in url

# Function to determine the type of URL: commit, pull, or none
def url_type(url):
    search = ["commit", "pull", 'issues', 'releases', 'security/advisories']
    for i in search:
        if i in url:
            return i
    return "none"

# Function to call the NIST API
def call_api(cwe_id, start_date, end_date):
    vulnerabilities = []
    start_index = 0
    results_per_page = 2000
    nr_fails = 0

    while True:
        params = {
            "cweId": cwe_id,
            "pubStartDate": start_date,
            "pubEndDate": end_date,
            "resultsPerPage": results_per_page,
            "startIndex": start_index
        }
        response = requests.get(BASE_URL, params=params)
        if response.status_code != 200:
            nr_fails += 1
            if nr_fails > 2:
                print(f"Failed to retrieve data after multiple attempts: {response.status_code} - {response.text}")
                break
            print(f"Failed to retrieve data: {response.status_code} - {response.text}")
            time.sleep(10)
            continue

        data = response.json()
        vulnerabilities.extend(data["vulnerabilities"])

        if start_index + results_per_page >= data["totalResults"]:
            break
        start_index += results_per_page

    return vulnerabilities

## Functions to process GITHUB data

In [ ]:
# Function to get commit info from GitHub API
def get_commit_info(owner, repo, commit_sha):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits/{commit_sha}"
    headers = {
        'Authorization': f'bearer {GITHUB_TOKEN}'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()
    return {
        "nr_files_changed": len(data['files']),
        "commit": commit_sha,
        "parent_commit": data['parents'][0]['sha'],
        "repo": f"{owner}/{repo}",
        "files": data['files'],
    }

# Function to get commit info from URL
def get_commit_info_from_url(url):
    owner, repo, commit_sha = parse_github_commit_url(url)
    return get_commit_info(owner, repo, commit_sha)

# Function to parse GitHub commit URL
def parse_github_commit_url(url):
    parsed_url = urlparse(url)
    path_parts = parsed_url.path.split('/')
    if len(path_parts) >= 5 and path_parts[3] == "commit":
        owner = path_parts[1]
        repo = path_parts[2]
        commit_sha = path_parts[4]
        return owner, repo, commit_sha
    else:
        raise ValueError("Invalid GitHub commit URL.")

# Support Function to gather data

In [ ]:
# Function to parse vulnerabilities and check for GitHub commits
def parse_vulnerability(vuln, cwe_id):
    cve = vuln['cve']
    cve_id = cve['id']
    published_date = cve.get('published', '')
    description = next((desc['value'] for desc in cve.get('descriptions', []) if desc['lang'] == "en"), None)
    references = cve.get('references', [])
    github_commits = [ref['url'] for ref in references if is_github(ref['url']) and url_type(ref['url']) == "commit"]

    if len(github_commits) != 1:
        return None

    commit_info = get_commit_info_from_url(github_commits[0])
    
    return {
        "nr_files_changed": commit_info['nr_files_changed'],
        "commit": commit_info['commit'],
        "parent_commit": commit_info['parent_commit'],
        "repo": commit_info['repo'],
        "files": json.dumps(commit_info['files']),
        'cve_id': cve_id,
        'cwe_id': cwe_id,
        'published_date': published_date,
        'description': description,
        'references': json.dumps(references),
        'github_commit': github_commits[0]
    }

# Function to scrape data between given dates for a specific CWE ID
def scrape_between(start_date: datetime, end_date: datetime, cwe_id: int) -> pd.DataFrame:
    date_ranges = generate_monthly_ranges_pd(start_date, end_date)
    all_vulnerabilities = []

    for start, end in date_ranges:
        vulnerabilities = call_api(cwe_id, start, end)
        print(f"Scraped {len(vulnerabilities)} vulnerabilities between {start} and {end}")
        for vuln in vulnerabilities:
            parsed_vuln = parse_vulnerability(vuln, cwe_id)
            if parsed_vuln:
                all_vulnerabilities.append(parsed_vuln)

    return pd.DataFrame(all_vulnerabilities)

# Function to generate monthly ranges
def generate_monthly_ranges_pd(start_date: datetime, end_date: datetime):
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    ranges = []

    if start.month == end.month and start.year == end.year:
        ranges.append((start.strftime("%Y-%m-%dT00:00:00.000"), end.strftime("%Y-%m-%dT23:59:59.999")))
    else:
        months_start = pd.date_range(start=start, end=end, freq='MS')
        months_end = pd.date_range(start=start, end=end, freq='ME')
        if end > months_end[-1]:
            months_end = months_end[:-1].append(pd.Index([end]))
        elif end < months_end[-1]:
            months_end = months_end[:-1]
            months_end = months_end.append(pd.Index([end]))

        ranges = [(start.strftime("%Y-%m-%dT00:00:00.000"), end.strftime("%Y-%m-%dT23:59:59.999")) for start, end in zip(months_start, months_end)]

    return ranges

# Function to load existing CSV data
def load_existing_data(filepath):
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        return pd.DataFrame()


# Main Execution

- As for usage of this repository it is important to define start and end date
- This enures smaller chunking of data as well as the code ensures that the data is not duplicated

In [26]:
# Define the start and end dates
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 10)

# Define the CWE IDs
cwe_ids = ["CWE-89", "CWE-79", "CWE-22"]

# Define the existing file name
existing_file = 'vulnerabilities.csv'

# Function to load existing data
def load_existing_data(file_name):
    try:
        return pd.read_csv(file_name)
    except FileNotFoundError:
        return pd.DataFrame()

# Load existing data
existing_data = load_existing_data(existing_file)
existing_keys = set(existing_data[['cwe_id', 'cve_id']].apply(tuple, axis=1)) if not existing_data.empty else set()

# Determine the starting vulnerability ID
if not existing_data.empty:
    last_vulnerability_id = existing_data['vulnerability_id'].max()
else:
    last_vulnerability_id = 0

all_dataframes = []
new_entries = []

for cwe_id in cwe_ids:
    print(f"Scraping data for {cwe_id} between {start_date} and {end_date}")
    df = scrape_between(start_date, end_date, cwe_id)
    if not df.empty:
        df['vulnerability_id'] = range(last_vulnerability_id + 1, last_vulnerability_id + 1 + len(df))
        new_entries_df = df[~df[['cwe_id', 'cve_id']].apply(tuple, axis=1).isin(existing_keys)]
        new_entries.extend(new_entries_df.to_dict(orient='records'))
        all_dataframes.append(new_entries_df)
        last_vulnerability_id += len(df)

if all_dataframes:
    combined_new_df = pd.concat(all_dataframes, ignore_index=True)
    if not existing_data.empty:
        combined_df = pd.concat([existing_data, combined_new_df], ignore_index=True)
    else:
        combined_df = combined_new_df
    combined_df.to_csv(existing_file, index=False)
    print("Data saved to vulnerabilities.csv")
else:
    print("No new entries found.")

Scraping data for CWE-89 between 2024-01-01 00:00:00 and 2024-01-10 00:00:00
Scraped 77 vulnerabilities between 2024-01-01T00:00:00.000 and 2024-01-10T23:59:59.999
Scraping data for CWE-79 between 2024-01-01 00:00:00 and 2024-01-10 00:00:00
Scraped 95 vulnerabilities between 2024-01-01T00:00:00.000 and 2024-01-10T23:59:59.999
Scraping data for CWE-22 between 2024-01-01 00:00:00 and 2024-01-10 00:00:00
Scraped 20 vulnerabilities between 2024-01-01T00:00:00.000 and 2024-01-10T23:59:59.999
Data saved to vulnerabilities.csv


In [27]:
combined_df.head()

,nr_files_changed,commit,parent_commit,repo,files,cve_id,cwe_id,published_date,description,references,github_commit,vulnerability_id
0,3,0d3d38cfa487481b66869e4212df1cefc281ecb7,30df154c87abc92765194272ece4c6e967b67b25,wp-plugins/rt-prettyphoto,"[{""sha"": ""9ecf9d11f9d16b4891361dabc87b983b6cd4...",CVE-2015-10128,CWE-79,2024-01-02T14:15:07.810,A vulnerability was found in rt-prettyphoto Pl...,"[{""url"": ""https://github.com/wp-plugins/rt-pre...",https://github.com/wp-plugins/rt-prettyphoto/c...,1
1,1,8d039d6efe80780adc40c6f670c06d21de272105,8717fbda818d97685f5498dd04bad86489262633,Zimbra/zm-ajax,"[{""sha"": ""de37d34b05afaddc47159c9701245fc20890...",CVE-2017-20188,CWE-79,2024-01-02T15:15:08.377,A vulnerability has been found in Zimbra zm-aj...,"[{""url"": ""https://github.com/Zimbra/zm-ajax/co...",https://github.com/Zimbra/zm-ajax/commit/8d039...,2
2,5,0df8a5e8722188744973168648e4c74c69ce67fd,be2dcd2ff34e11d0ed26baa01cd5e2ff20f092b8,acumos/design-studio,"[{""sha"": ""5292e807495a13ffdca2304d44cf08a6551d...",CVE-2018-25097,CWE-79,2024-01-02T16:15:11.100,"A vulnerability, which was classified as probl...","[{""url"": ""https://github.com/acumos/design-stu...",https://github.com/acumos/design-studio/commit...,3
3,1,c3d78b7e49f5fe49a9d07725c3174d005deaa597,73cfb44666818eefd501b526a894fe884dd12129,PrestaShop/PrestaShop,"[{""sha"": ""4f4cea68c999daae11c1d135446746bc09a8...",CVE-2024-21628,CWE-79,2024-01-02T22:15:09.687,PrestaShop is an open-source e-commerce platfo...,"[{""url"": ""https://github.com/PrestaShop/Presta...",https://github.com/PrestaShop/PrestaShop/commi...,4
4,1,0f36e2f521ade8ddfb3e04786defe074370afb50,6656de201efe67c7983102c344a546eed976a819,wp-sms/wp-sms,"[{""sha"": ""6f7839dcf0df293f224cd3403bdd947330cf...",CVE-2023-6980,CWE-79,2024-01-03T06:15:47.500,The WP SMS – Messaging & SMS Notification for ...,"[{""url"": ""https://github.com/wp-sms/wp-sms/com...",https://github.com/wp-sms/wp-sms/commit/0f36e2...,5
5,1,6656de201efe67c7983102c344a546eed976a819,08aca53fb88ac0d2e31a7f9525d10ba2430176f1,wp-sms/wp-sms,"[{""sha"": ""71f2ae90c4d82913d9fec4beaa479d0e5bed...",CVE-2023-6981,CWE-89,2024-01-03T06:15:47.663,The WP SMS – Messaging & SMS Notification for ...,"[{""url"": ""https://github.com/wp-sms/wp-sms/com...",https://github.com/wp-sms/wp-sms/commit/6656de...,6
6,10,d348c43b24a9de350ff6e5bd610545a10c1fc712,e5c88ece1b7881879ed1a6f65e5f81d3cf788c4c,iBotPeaches/Apktool,"[{""sha"": ""5c9e3de4a269d2b023c843aa07c21752d7f8...",CVE-2024-21633,CWE-22,2024-01-03T17:15:13.103,Apktool is a tool for reverse engineering Andr...,"[{""url"": ""https://github.com/iBotPeaches/Apkto...",https://github.com/iBotPeaches/Apktool/commit/...,13
7,14,5558233fb7defda706b4f9c87c17759705949889,8a816db15844dd4b3d2c7efa89c76bdd5e9e6f74,boazsegev/iodine,"[{""sha"": ""616475bca840f9d0f1e66b57938addc3ddaf...",CVE-2024-22050,CWE-22,2024-01-04T21:15:10.100,Path traversal in the static file service in I...,"[{""url"": ""https://github.com/advisories/GHSA-8...",https://github.com/boazsegev/iodine/commit/555...,14
